<a href="https://colab.research.google.com/github/jesux009/03MIAR-Algoritmos-de-Optimizaci-n/blob/main/Trabajo_Pr%C3%A1ctico_Algoritmos_de_Optimizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random
import math
import matplotlib.pyplot as plt

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Jesús Morales López  <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---/tree/master/TrabajoPractico<br>
Google Colab: https://colab.research.google.com/drive/1XC08ua3ZjAL94kb4AGZRmidCiS4pcUjg?usp=sharing <br>

**Problema:**

Organizar los horarios de partidos de una jornada de La Liga<br>

Descripción del problema:

Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.

Los horarios disponibles se conocen a priori y son los siguientes:
- Viernes a las 20h
- Sábado a las 12h, a las 16h, a las 18h y a las 20h
- Domingo a las 12h, a las 16h, a las 18h y a las 20h
- Lunes a las 20h

En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores (que tiene relación directa con la audiencia). Hay 3 equipos en la categoría A, 11 equipos de categoría B y 6 equipos de categoría C. Se conoce estadísticamente la audiencia que genera cada partido según los equipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos). Las categorías de los equipos son:

**Categoría A**
- Real Madrid
- Real Sociedad
- Barcelona

**Categoría B**
- Celta
- Valencia
- Athletic
- Villarreal
- Alavés
- Levante
- Espanyol
- Sevilla
- Betis
- Atlético
- Getafe

**Categoría C**
- Mallorca
- Eibar
- Leganés
- Osasuna
- Valladolid
- Granada

Los espectadores en enfrentamientos por categoría, si se televisa la noche del sábado (horario de máxima audiencia) son:

- A vs. A = 2M
- A vs. B = 1.3M
- A vs. C = 1M
- B vs. B = 0.9M
- B vs. C = 0.75M
- C vs. C = 0.47M

Si el horario del partido no se realiza a las 20 horas del sábado se sabe que se reduce según los coeficientes de la siguiente tabla:

| Hora | Viernes | Sábado | Domingo | Lunes |
|------|---------|--------|---------|-------|
| 12h  |   -     | 0.55   | 0.45    |   -   |
| 16h  |   -     | 0.7    | 0.75    |   -   |
| 18h  |   -     | 0.8    | 0.85    |   -   |
| 20h  |  0.4    | 1      | 1       | 0.4   |

Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes.

Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada yse estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias:

| Coincidencias | - % |
|---------------|-----|
| 0 | - 0% |
| 1 | - 25% |
| 2 | - 45% |
| 3 | - 60% |
| 4 | - 70% |
| 5 | - 75% |
| 6 | - 78% |
| 7 | - 80% |
| 8 | - 80% |

La jornada a optimizar es la que enfrenta a los siquientes equipos:
- Celta vs. Real Madrid
- Valencia vs. Real Sociedad
- Mallorca vs. Eibar
- Athletic vs. Barcelona
- Leganés vs. Osasuna
- Villarreal vs. Granada
- Alavés vs. Levante
- Espanyol vs. Sevilla
- Betis vs. Valladolid
- Atlético vs. Getafe
  

# Modelo
- **¿Como represento el espacio de soluciones?**
- **¿Cual es la función objetivo?**
- **¿Como implemento las restricciones?**

El problema consiste en asignar los 10 partidos de una jornada de La Liga a los 10 horarios disponibles, con el objetivo de maximizar la audiencia total.

Empezaremos definiendo la variable binaria $x_{i,h}$:

$$
\begin{align}
  x_{i,h} = \begin{cases}
          1, \text{ si el partido i se asigna al horario h}\\
          0, \text{ en caso contrario}
        \end{cases}
\end{align}
$$

El índice $i$ para representar los partidos se ha definido en base a indexar el ejemplo visto en clase, aunque se podría elegir cualquier otra ordenación. Por lo tanto $i \in I, I = \{0...9\}$. La correspondencia entre índices y encuentros se recoge a continuación:

- 0: Celta vs. Real Madrid
- 1: Valencia vs. Real Sociedad
- 2: Mallorca vs. Eibar
- 3: Athletic vs. Barcelona
- 4: Leganés vs. Osasuna
- 5: Villarreal vs. Granada
- 6: Alavés vs. Levante
- 7: Espanyol vs. Sevilla
- 8: Betis vs. Valladolid
- 9: Atlético vs. Getafe

A su vez, el índice $h$ para representar cada franja horaria tomará valores $h \in H, H = \{0...9\}$, correspondiendose con:
- 0: Viernes, 20:00
- 1: Sábado, 12:00
- 2: Sábado, 16:00
- 3: Sábado, 18:00
- 4: Sábado, 20:00
- 1: Domingo, 12:00
- 2: Domingo, 16:00
- 3: Domingo, 18:00
- 4: Domingo, 20:00
- 9: Lunes, 20:00

Cabe notar que la notación podría haber sido distinta, por ejemplo usando para los horarios una abreviatura que hiciese más fácil interpretar las soluciones ($V20, S12, S16, ...$). Sin embargo, ya que se va a usar Python para implementar el algoritmo, es sencillo crear una función que mapee / tabule el óptimo encontrado una vez se llegue al criterio de parada, y mientras tanto hacer uso de cómo Python indexa sus listas para representar de manera sencilla las constantes del sistema.

Las restricciones descritas por el problema son las siguientes:
- Cada partido debe jugarse exactamente en un horario. Esto garantiza que no haya partidos sin horario o partidos en múltiples franjas horarias.
$$\sum_{h \in H} x_{i, h} = 1$$
- Se televisará un partido de manera obligatoria los viernes y los lunes.
$$ \sum_{i \in I} x_{i, 0} = 1 $$
$$ \sum_{i \in I} x_{i, 9} = 1 $$
- Existe además una restricción "blanda" sobre el número de partidos que se juegan en la misma franja. Es posible solapar partidos aunque esto conlleve una penalización sobre el número de espectadores. Se $n_h$ como el número de partidos en la franja $h \in H$, por lo que $n_h = \sum_{i \in I} x_{i,h}$. Por lo tanto, $p(n_h)$ representará la penalización causada por tener partidos en la misma franja horaria.

Para representar formalmente el espacio de soluciones se define una estructura basada en un vector de asignación. Aunque técnicamente es posible aplicar algoritmos solamente con la variable binaria, en cuanto a la implementación de la algoritmia es mucho más natural y cómodo trabajar con esta representación al simplificar acciones como explorar vecindarios (en búsqueda local) o realizar mutaciones (en algoritmos genéticos). Se define por lo tanto un vector solución:

$$
S = \{ h_0, h_1, h_2, h_3, h_4, h_5, h_6, h_7, h_8, h_9\}
$$

donde cada $h_i$ representa el horario $h$ asignado al partido $i$. Se observa por lo tanto cómo la variable $x_{i,h}$ se relaciona con $h_i$ de la siguiente manera:

$$
h_i = \sum_{h \in H} h \cdot x_{i, h}
$$

Así pues, la solución ejemplo de la práctica es $S= \{ 0, 1, 2, 3, 4, 6, 6, 7, 8, 9\}$.

También se podría haber definido al revés, representando cada posición del vector como una franja horaria e indicando en cada uno los partidos que se colocan ahí, lo cual resulta más intuitivo. Sin embargo, la notación se complicaría para poder representar múltiples partidos en la misma franja y no tiene la mayor ventaja que tiene esta representación: la restricción #1 se aplica por defecto puesto que todos los índices tendrán un sólo valor asignado.

Una vez hemos definido todos estos componentes, podemos pasar a definir la función objetivo, que no será otra que el número de espectadores totales de la jornada, la cual habrá que maximizar. Definimos por lo tanto esta función objetivo $F$ como:
$$
F = \sum_{i \in I} \sum_{h \in H} A_i \cdot k_h \cdot [1-p(n_h)] \cdot x_{i,h}
$$

donde:
- $A_i$ = Audiencia base del partido si se emitiera en horario de máxima audiencia.
- $k_h$ = Coeficiente multiplicador del horario $h$.
- $p(n_h)$ = Penalización por tener varios partidos en la misma franja horaria, como ya se definió anteriormente.

In [2]:
## Empezamos definiendo las constantes del sistema
# Audiencia para cada partido, en millones de espectadores
A = np.array([
    1.30, # Celta vs. R. Madrid      (B vs. A)
    1.30, # Valencia vs. R. Sociedad (B vs. A)
    0.47, # Mallorca vs. Eibar       (C vs. C)
    1.30, # Athletic vs. Barcelona   (B vs. A)
    0.47, # Leganés vs. Osasuna      (C vs. C)
    0.75, # Villarreal vs. Granada   (B vs. C)
    0.90, # Alavés vs. Levante       (B vs. B)
    0.90, # Espanyol vs. Sevilla     (B vs. B)
    0.75, # Betis vs. Valladolid     (B vs. C)
    0.90  # Atlético vs. Getafe      (B vs. B)
], dtype=np.float16)
# Coeficiente multiplicador del horario
k = np.array([
    0.40, # Viernes a las 20h
    0.55, # Sábado  a las 12h
    0.70, # Sábado  a las 16h
    0.80, # Sábado  a las 18h
    1.00, # Sábado  a las 20h
    0.45, # Domingo a las 12h
    0.75, # Domingo a las 16h
    0.85, # Domingo a las 18h
    1.00, # Domingo a las 20h
    0.40  # Lunes   a las 20h
], dtype=np.float16)
# Penalización por partidos coincidentes
p = np.array([
    0.00, # 0 partidos coincidentes
    0.25, # 1 partidos coincidentes
    0.45, # 2 partidos coincidentes
    0.60, # 3 partidos coincidentes
    0.70, # 4 partidos coincidentes
    0.75, # 5 partidos coincidentes
    0.78, # 6 partidos coincidentes
    0.80, # 7 partidos coincidentes
    0.80  # 8 partidos coincidentes
], dtype=np.float16)

# Definiremos una función que valide si una solución es válida:
def validar_solucion(solucion: list|np.ndarray, A: list|np.ndarray,  k: list|np.ndarray):
  """
  Valida si una solución es válida, es decir, cumple con las restriccioens del problema:
  - Cada partido debe jugarse exactamente en un horario.
  - Se televisará un partido de manera obligatoria los viernes (0) y los lunes (9).

  Argumentos:
    - solucion (list | np.array): Lista que contiene, para cada índice (partido i), el horario asignado (h).
    - A (list | np.array): Lista que contiene, para cada índice (partido i), la audiencia base del partido.
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.

  Devuelve:
    - bool: True si la solución es válida, False en caso contrario.
  """
  if (len(solucion) == len(A)) and (0 in solucion) and (len(k)-1 in solucion) and np.all((solucion >= 0) & (solucion <= len(k)-1)):
    return True
  else:
    return False

# A continuación, definiremos una función que traslade una solución en formato vectorial a una matriz con los valores de las variables binarias
def solucion_a_binario(solucion: list|np.ndarray, k: list|np.ndarray):
  """
  Convierte una solución en formato vectorial a una matriz con los valores de las variables binarias.
  La matriz tendrá un tamaño nxm, siendo n el número de partidos y m el número de franjas horarias.
  La variable binaria x_{ih} tendrá un valor de 1 si el partido i se asigna al horario h, y 0 en caso contrario.

  Argumentos:
    - solucion (list | np.array): Lista que contiene, para cada índice (partido i), el horario asignado (h).
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.

  Devuelve:
    - solucion_binario (np.array(np.array)): Matriz con los valores de las variables binarias x_{ih} para la solución dada.
  """
  # Creamos primero la matriz con las dimensiones correctas pero nula
  solucion_bin = np.zeros((len(solucion), len(k)), dtype=np.bool_)
  # Iteramos la lista y asignamos 1 al valor de la fila en la que nos encontremos y la columna que indique el valor en solución
  for i in range(len(solucion)):
    solucion_bin[i, solucion[i]] = 1
  return solucion_bin


# Construimos también una función que, tomando la solución en variables binarias, devuelva la penalización a aplicar a cada horario
def penalizacion_horario(solucion_bin: list|np.ndarray, p: list|np.ndarray):
  """
  Calcula la penalización a aplicar a cada horario para una solución expresada en variables binarias.

  Argumentos:
    - solucion_bin (list(list) | np.array(np.array)): Matriz con los valores de las variables binarias x_{ih}.
    - p (list | np.array): Lista que contiene, para cada índice (número de partidos coincidentes n), la penalización asociada.

  Devuelve:
    - penalizacion (np.array): Lista que contiene, para cada índice (horario h), la penalización asociada a los partidos que se jueguen en ese horario.
  """
  # Inicializamos el vector resultado que contendrá la penalizacion para cada horario
  penalizacion = np.zeros(solucion_bin.shape[1], dtype=np.float16)
  # Calculamos el número de partidos coincidentes para cada horario
  n_h = np.maximum(0, np.sum(solucion_bin, axis=0) - 1)
  # Rellenamos el resultado de acuerdo al número de coincidencias
  for i in range(len(n_h)):
    penalizacion[i] = p[n_h[i]]
  return penalizacion


# Finalmente, componemos una función que calcule los espectadores totales a partir de una solución en forma de lista
def espectadores_totales(solucion: list|np.ndarray, A: list|np.ndarray, k: list|np.ndarray, p: list|np.ndarray):
  """
  Calcula los espectadores totales a partir de una cierta solución al problema de asignación de partidos a franjas horarias.
  La función está preparada para calcular los espectadores de una solución parcial (solución con menos valores que longitud tiene A.

  Argumentos:
    - solucion (list | np.array): Lista que contiene, para cada índice (partido i), el horario asignado (h).
    - A (list | np.array): Lista que contiene, para cada índice (partido i), la audiencia base del partido.
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.
    - p (list | np.array): Lista que contiene, para cada índice (número de partidos coincidentes n), la penalización asociada.

  Devuelve:
    - espectadores (float): Espectadores totales generados por esta asignación, en millones.
  """
  # Inicializamos el conteo de espectadores
  espectadores = 0
  # Primero convertimos la solución en forma de lista a la matriz de variables binarias
  solucion_bin = solucion_a_binario(solucion, k)
  # Calculamos, para cada horario, la penalización recibida por solapar partidos
  penalizacion = penalizacion_horario(solucion_bin, p)
  # Finalmente, calculamos el total con los datos de la solución.
  if len(solucion) == len(A):
    # Cálculo hecho mediante broadcasting en numpy para acelerar cálculos futuros
    espectadores = np.sum(A[:, None] * k * (1 - penalizacion) * solucion_bin)
  else:
    # En este caso tomamos los valores base de A solo para los partidos asignados
    A_parcial = A[:len(solucion)]
    espectadores = np.sum(A_parcial[:, None] * k * (1 - penalizacion) * solucion_bin)
  return espectadores


# Ejemplo de prueba con la solución de la transparencia #39
solucion_ejemplo = np.array([0,1,2,3,4,6,6,7,8,9])
print(f'Solución ejemplo: {solucion_ejemplo}')
print(f'¿Es la solución valida?: {validar_solucion(solucion_ejemplo, A, k)}')
print(f'Solución en forma binaria:')
print(solucion_a_binario(solucion_ejemplo, k))
print(f'Espectadores totales: {espectadores_totales(solucion_ejemplo, A, k, p)}M')

Solución ejemplo: [0 1 2 3 4 6 6 7 8 9]
¿Es la solución valida?: True
Solución en forma binaria:
[[ True False False False False False False False False False]
 [False  True False False False False False False False False]
 [False False  True False False False False False False False]
 [False False False  True False False False False False False]
 [False False False False  True False False False False False]
 [False False False False False False  True False False False]
 [False False False False False False  True False False False]
 [False False False False False False False  True False False]
 [False False False False False False False False  True False]
 [False False False False False False False False False  True]]
Espectadores totales: 5.875M


#Análisis
- **¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones**

El problema planteado pertenece a la familia de problemas de optimización combinatoria en planificación, conel objetivo de maximizar el beneficio.

Aunque la asignación básica partido-horario podría parecer un problema de asignación clásico, aquí aparecen dos elementos que lo hacen computacionalmente difícil:
- Principalmente, el hecho de poder asignar a una misma franja varios partidos, incrementa enormemente el espacio de soluciones, como se comentará a continuación.
- A consecuencia del punto anterior, se describe que la audiencia de un partido no depende solo de su horario sino también del número de partidos que suceden en ese horario. Por lo tanto, aparecen interacciones como $x_{i,h}$ ↔ $x_{j,h}$. La decisión de asignar un partido afecta al beneficio de otros. Sobra decir en este caso que estas relaciones entre las variables hacen que la función objetivo ya no sea lineal.

Debido a estas características se puede clasificar como un problema *NP-hard*. De hecho, se trata de una versión del *Quadratic Assignment Problem* (QAP), el cual se ha demostrado ser *NP-hard*. Por lo tanto, **no existe un algoritmo que sea capaz de encontrar una solución óptima en un tiempo polinómico.** Por lo tanto, se deberán usar técnicas de optimización metaheurísticas para encontrar una solución suficientemente buena.

En cuanto al espacio de soluciones, se puede calcular fácilmente teniendo en cuenta que:
- El viernes se debe colocar un partido obligatoriamente, donde hay 10 opciones a elegir
- El lunes sucede algo parecido, esta vez con 9 opciones (una se ha asignado seguro al viernes)
- Para el resto de horarios (8 en total), quedan 8 partidos que asignar, los cuales, al permitirse solapamientos, no siguen con esa progresión factorial sino que en total existirán $8^8$ posibilidades.

En total, el espacio de soluciones estará compuesto por:
$$
|S| = 10 \cdot 9 \cdot 8^8 \approx 1.51\cdot 10^9 = 1.51 \text{ mil millones de soluciones posibles}
$$
El orden del espacio de soluciones del problema es, por lo tanto, exponencial.

Nótese que si no existiesen solapamientos, el total sería de $|S| = 10! = 3,628,800$ soluciones, lo cual es alto abordable y se podría resolver de manera exacta por fuerza bruta. En general, se podrían aplicar algortimos de optimización con orden polinómico, puesto que se reduciría a un problema de asignación clásico.

#Diseño
- **¿Que técnica utilizo? ¿Por qué?**

Como se ha analizado previamente, el problema presenta:

- Un espacio de soluciones exponencial
- Una función objetivo no lineal
- Dependencias globales entre variables

Estas características hacen inviable el uso de fuerza bruta, programación lineal clásica o métodos exactos. Por ello, la manera más eficiente de abordar el problema es mediante técnicas metaheurísticas, que permiten explorar espacios de búsqueda muy grandes obteniendo soluciones cercanas al óptimo en tiempos más asumibles.

Para resolver el problema, se ha optado por usar el algoritmo GRASP (Greedy Randomized Adaptative Search). Este algoritmo combina la construcción voraz (intuitivamente útil en este problema) y la aleatorización para poder diversificar. Así, en la fase constructiva se pueden asignar primero los partidos de mayor audiencia potencial a las mejores franjas (con cierta aleatoriedad includida), y posteriormente, una búsqueda local refina cada solución. Esto permite obtener buenas soluciones iniciales de forma rápida y explorar distintas regiones del espacio sin necesidad de recorrerlo exhaustivamente.

Además, ayuda a la elección el no haber tenido la oportunidad de codificar este algortimo durante las sesiones de la asignatura, por lo que se aprovecha la oportunidad para realizar la demostración.

## GRASP (Greedy Randomized Adaptative Search Procedure)

El desarrollo del método que se recoge a continuación es bastante estándar para un algoritmo GRASP. A continuación se recoge una descripción a alto nivel de la arquitectura del algoritmo, destacando y justificando algunas elecciones en el diseño.

El primer paso de cualquier algoritmo GRASP es la elaboración de una función que genere una solución válida del problema con conceptos de voracidad (mejora de la solución inicial) y aleatoriedad (diversificación). Para ello, es común construir una lista restringida de candidatos (LRC) obtenida a partir de técnicas voraces y seleccionar una de las opciones al azar con continuar construyendo la solución parcial. La función `construccion_voraz` hace exactamente eso: para cada valor del vector solución, se construye una lista con los $n$ (controlable mediante `n_LRC`) candidatos que más maximicen la audiencia para esa elección, y se escoge uno de ellos al azar. Se podría haber tomado otro criterio, como calcular un umbral para la LRC, pero tras realizar pruebas se ha observado que los valores del multiplicador de audiencia según horario no se prestan a este criterio (grandes saltos entre horarios) y la eficacia de la aproximación actual ha sido mejor. De la misma manera, para el tamaño del problema, se ha encontrado que hay un buen equilibrio entre voracidad y aleatoriedad con un `n_LRC=3`.

Una vez se tiene la solución inicial, se exploran posibilidades cercanas mediante un algoritmo de búsqueda local. Para ello, se ha construido primero una función `genera_vecina` que genera una solución "vecina" a la dada realizando una de dos operaciones: cambio de un partido a otra franja horaria (1-opt) o intercambio de partidos entre franjas horarias (2-opt). Es necesario que ambos tipos de vecinos se generen por varios motivos:
- Se comprueba que las penalizaciones por solapamiento tienden a empeorar las soluciones, por lo que para poder explorar soluciones sin solapes, es necesario que se exploren vecinas generadas a partir del intercambio de horarios entre dos partidos (2-opt).
- Aunque un análisis preliminar indica que el solapamiento de partidos no es óptimo, se deberían explorar este tipo de vecinos para comprobar si existe alguna solución que aproveche esta flexibilidad. Por lo tanto, si se parte de una solución donde todos los horarios tienen un partido, la única manera de explorar solapamientos sería cambiando un partido de horario (1-opt).

En el proceso de búsqueda local, no se ha incluido un criterio de parada al uso, sino que se deja la posibilidad de generar aleatoriamente un número de vecinos `max_iter_vecinas` que no sean capaz de mejorar la solución de su antecesor antes de frenar el proceso de búsqueda. Nótese que para las dimensiones del ejercicio, sería posible generar todas las vecinas del problema en esta función y realizar la comparación solamente con la mejor. Sin embargo, se ha considerado esta aproximación porque la alternativa no garantiza que se acabe llegando a un mejor óptimo local (puede que esa solución vecina obtenga más espectadores que las demás pero sus vecinas no la mejoren, mientras que cogiendo alguna otra del espacio de vecinas se consiga progresar entre vecinas para llegar a una solución mejor incluso). La estrategia es, por lo tanto, permitir la generación aleatoria de soluciones vecinas, parando la búsqueda local tras un número de intentos razonable, y actualizando la mejor solución del algoritmo de búsqueda local en cuanto alguna vecina mejore la solución óptima actual. En el proceso de generación de vecinas, se ha usado una distribución binomial, donde 2 de cada 3 veces se prueba una transformación 2-opt, y el resto una 1-opt. Nótese que aquellas vecinas que no cumplan con las restricciones del problema son inmediatamente rechazadas y no se tienen en cuenta para el conteo de `max_iter_vecinas`.

Estos dos pasos se repiten mediante re-arranques (reseteos de la solución inicial voraz-aleatoria), controlables en la función `grasp` mediante el parámetro `num_iter`.

Finalmente, se ha usado todo lo posible la librería `numpy` para optimizar el tiempo de cálculo y aprovechar caracerísticas y funciones propias.

In [16]:
## Algoritmo GRASP

# Empezamos construyendo una función que genere soluciones voraces con un toque de aleatoriedad, pero cumpla las restricciones
def construccion_voraz(A: list|np.ndarray, k: list|np.ndarray, n_LRC:int=3):
  """
  Genera una solución voraz para el problema de asignación de partidos a franjas horarias, con un toque de aleatoriedad según el concepto de GRASP.
  La solución voraz considera asignar partidos de mayor audiencia a las franjas con mayor audiencia potencial, sin solapes.

  Argumentos:
    - A (list | np.array): Lista que contiene, para cada índice (partido i), la audiencia base del partido.
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.
    - n_LRC (int): Número de horarios que se integrarán en la lista restringida de candidatos. Cuanto menor sea, menos componente aleatorio habrá

  Retornos:
    - solucion_voraz (np.array): Solución con forma de lista, que contiene para cada índice (partido i), el horario asignado (h).
  """
  # Inicializamos la solución, marcando con un -1 los no asignados
  solucion_voraz = -np.ones(len(A), dtype=np.int8)
  # Ordenamos los partidos por audiencia base descendente y los horarios por su coeficiente multiplicador ascendente
  orden_partidos = np.argsort(-A)
  orden_horarios = np.argsort(-k)
  # Iteramos a lo largo de la lista de partidos en el orden en el que más espectadores tienen.
  # Así, daremos prioridad a esos partidos en mejores horarios.
  for i in range(len(orden_partidos)):
    # Generamos la lista restringida de candidatos con las n opciones más voraces para el horario
    LRC = orden_horarios[:n_LRC] if len(orden_horarios) > n_LRC else orden_horarios
    # Escogemos aleatoriamente un horario de la LRC
    h_seleccionado = random.choice(LRC)
    # Asignamos este horario al partido
    solucion_voraz[orden_partidos[i]] = h_seleccionado
    # Quitamos esta opción de la lista posible de horarios. Esto se debe a que en la mayoría de casos, la solución más voraz es evitar solapes.
    orden_horarios = np.delete(orden_horarios, np.where(orden_horarios == h_seleccionado))
  return solucion_voraz


# Definimos una función para generar una solución "vecina". Se introducirán dos posibilidades para la definición de vecinos: cambio de franja para un partido (1-opt) e intercambio de partidos (2-opt).
def genera_vecina(solucion: list|np.ndarray, k: list|np.ndarray, mode:str='1-opt'):
  """
  Genera una solución vecina a la propuesta. Para la generación de esta vecina, se pueden usar dos métodos: cambio de franja para un partido (1-opt) e intercambio de partidos (2-opt).
  Nótese que esta vecina no tiene por qué mejorar la puntuación anterior. Para más información, consultar la sección descriptiva 'Diseño'.

  Argumentos:
    - solucion (list | np.array): Lista que contiene, para cada índice (partido i), el horario asignado (h).
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.
    - mode (str): Modo de búsqueda local. Si se indica '1-opt' el algoritmo reasignará un partido a otro horario. Si se indica '2-opt' el algoritmo intercambiará dos partidos. Se recomienda el uso esporádico de '1-opt' por defecto para explorar solapes.

  Retornos:
    - solucion_vecina (list | np.array): Lista generada, 'vecina' a la introducida, que contiene para cada índice (partido i), el horario asignado (h).
  """
  # Inicializamos la solución vecina a partir de la original
  solucion_vecina = np.array(solucion).copy()

  # MÉTODO '1-opt': Implementación para generar una vecina con una reasignación de horario
  if mode == '1-opt':
    # Elegimos un partido aleatorio de entre los posibles
    i = random.randint(0, len(solucion) - 1)
    # Generamos una lista con las
    franjas_disponibles = [h for h in range(len(k)) if h != solucion[i]]
    # Elegimos nueva franja distinta de forma aleatoria
    nueva_franja = random.choice(franjas_disponibles)
    # Asignamos esa nueva franja al partido seleccionado
    solucion_vecina[i] = nueva_franja
    return solucion_vecina

  # MÉTODO '2-opt': Implementación para generar una vecina con un intercambio de horarios
  elif mode == '2-opt':
    # Se seleccionan dos partidos aleatorios
    i, j = random.sample(range(len(solucion)), 2)
    # Se intercambian sus horarios
    solucion_vecina[i], solucion_vecina[j] = solucion[j], solucion[i]
    return solucion_vecina

  # Si el modo de búsqueda no es correcto, se devuleve un error
  else:
    raise ValueError("El parámetro 'mode' debe ser un string de entre las opciones '1-opt' (cambio de franja) o '2-opt' (intercambio de horarios)")


# Ahora definimos el paso de búsqueda local para la solución.
def busqueda_local(solucion: list|np.ndarray, A:list|np.ndarray, k:list|np.ndarray, p:list|np.ndarray, max_iter_vecinas:int=200, prob_2opt:float=0.667):
  """
  Busca una solución optimizada a partir de una solución inicial por medio de un algoritmo de búsqueda local.
  Este algoritmo permite dos definiciones de vecindario: cambio de franja de un partido (1-opt), e intercambio de partidos (2-opt)

  Argumentos:
    - solucion (list | np.array): Lista que contiene, para cada índice (partido i), el horario asignado (h).
    - A (list | np.array): Lista que contiene, para cada índice (partido i), la audiencia base del partido.
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.
    - p (list | np.array): Lista que contiene, para cada índice (número de partidos coincidentes n), la penalización asociada.
    - max_iter_vecinas (int): Iteraciones seguidas máximas que se realizarán en la búsqueda de una mejor vecina a la solución actual.
    - prob_2opt (str): Proporción de cambios tipo '2-opt' a generar en la búsqueda de vecinos ('2-opt' explora el intercambio de dos partidos). La alternativa es '1-opt', donde el algoritmo reasignará un partido a otro horario. Se recomienda el uso esporádico de '1-opt' por defecto para explorar solapes.

  Retornos:
    - solucion_optimizada (list | np.array): Lista optimizada a partir de la solución inicial usando un algoritmo de búsqueda local.
    - iteraciones (int): Número de iteraciones que se han realizado en la búsqueda local.
  """
  # Inicializamos la solucion inicial
  mejor_sol = np.array(solucion).copy()
  mejor_espectadores = espectadores_totales(mejor_sol, A, k, p)
  # Inicializamos las iteraciones que se usarán para la búsqueda de mejores vecinos
  iter = 0
  iter_global = 0
  # Se continúa con el bucle hasta que no haya soluciones vecinas mejores.
  # NOTA: Se ha decidido explorar vecinas aleatorias dadas las grandes dimensiones del espacio de soluciones global. Se introduce un criterio de parada de máximas iteraciones cuando no haya mejoras en soluciones vecinas.
  # Para más información, consultar la sección descriptiva 'Diseño'.
  while True:
    # Se decide aleatoriamente la vecina a explorar. Puesto que los solapes suelen ser menos óptimos (y además pueden generar soluciones no válidas por la restricción de lunes/viernes), este modo tiene menos posibilidades de aparecer.
    mode = '2-opt' if np.random.binomial(1, prob_2opt) else '1-opt'
    # Se genera y evalúa una vecina
    solucion_vecina = genera_vecina(mejor_sol, k, mode)
    espectadores_vecina = espectadores_totales(solucion_vecina, A, k, p)
    # Sólo contamos con las soluciones válidas
    if validar_solucion(solucion_vecina, A, k):
      # Si la solución vecina mejora la anterior, se acepta como solución óptima y se resetea el número de iteraciones sin mejora
      if espectadores_vecina > mejor_espectadores:
        mejor_sol = solucion_vecina
        mejor_espectadores = espectadores_vecina
        iter = 0
      # Si la solución vecina no mejora a la anterior, se contabiliza la iteración
      else:
        iter += 1
    # Ignoramos soluciones no válidas
    else:
      continue
    # Criterio de parada en la búsqueda de mejores vecinas
    if iter == max_iter_vecinas:
      break
    iter_global += 1
  return mejor_sol, mejor_espectadores


def grasp(A:list|np.ndarray, k:list|np.ndarray, p:list|np.ndarray, num_iter:int=50):
  """
  Aplica el algoritmo GRASP para encontrar una solución óptima al problema de asignación de partidos a franjas horarias.

  Argumentos:
    - A (list | np.array): Lista que contiene, para cada índice (partido i), la audiencia base del partido.
    - k (list | np.array): Lista que contiene, para cada índice (horario h), el coeficiente de multiplicación del mismo.
    - p (list | np.array): Lista que contiene, para cada índice (número de partidos coincidentes n), la penalización asociada.
    - num_iter (int): Número de rearranques (reseteos de la construcción de la solución inicial).

  Retornos:
    - mejor_sol (list | np.array): Lista optimizada a partir de la solución inicial usando GRASP.
    - mejor_espectadores (float): Espectadores totales generados por esta asignación, en millones.
  """
  # Inicializamos los espectadores
  mejor_espectadores = -999
  # Se repite el algoritmo de búsqueda local mejorado con solución voraz un número definido de veces
  for i in range(num_iter):
    # PASO 1. Construcción de una solución voraz aleatorizada
    solucion_voraz = construccion_voraz(A, k)
    # PASO 2. Búsqueda local para la solución inicial construida
    solucion, espectadores = busqueda_local(solucion_voraz, A, k, p)
    # Si esta solución mejora la solución global, se guardan los datos
    if espectadores > mejor_espectadores:
      mejor_solucion = solucion
      mejor_espectadores = espectadores
  return mejor_solucion, mejor_espectadores

solucion, espectadores = grasp(A, k, p)
print(f'Solución: {solucion}')
print(f'Espectadores totales: {espectadores}M')


Solución: [4 8 0 7 9 1 3 6 5 2]
Espectadores totales: 6.85546875M


Por lo tanto el horario final es:
| Horario              | Partido                                     |
|----------------------|---------------------------------------------|
| Viernes, 21:00       | Mallorca vs. Eibar                          |
| Sábado, 12:00        | Villarreal vs. Granada                      |
| Sábado, 16:15        | Atlético vs. Getafe                         |
| Sábado, 18:30        | Alavés vs. Levante                          |
| Sábado, 21:00        | Celta vs. Real Madrid                       |
| Domingo, 12:00       | Betis vs. Valladolid                        |
| Domingo, 16:15       | Espanyol vs. Sevilla                        |
| Domingo, 18:30       | Athletic vs. Barcelona                      |
| Domingo, 21:00       | Valencia vs. Real Sociedad                  |
| Lunes, 21:00         | Leganés vs. Osasuna                         |

Se observa que se ha optado por no solapar partidos (aunque la posisbilidad estaba disponible). Además, se observa que la solución de espectadores final (6.86 millones aproximadamente) se obtiene con varios horarios, como por ejemplo con $[8 \: 7 \: 9 \: 4 \: 0 \: 5 \: 3 \: 6 \: 1\: 2]$.

# Conclusiones

Durante el transcurso de este trabajo práctico, se ha llevado a cabo la descripción del problema a resolver (organización de los horarios de una jornada de la liga española de fútbol), aportando observaciones que ayudsn a su resolución, tanto desde un punto de vista descriptivo (intuición de que voracidad debe funcionar bien en el problema, análisis de las retricciones y penalizaciones) y matemático (formalización del tamaño del espacio de soluciones y clasificación del problema).

Posteriormente, se ha justificado y llevado a cabo la elcción de un algoritmo de optimización para maximizar la audiencia de la jornada, decidiendo usar el algoritmo GRASP dado su equilibrio entre voracidad, intensificación mediante búsqueda local y diversificación mediante el uso de aleatoriedad en la generación de soluciones iniciales.

Tras múltiples simulaciones, se ha obtenido que el máximo de expectadores esperados está alrededor de los 6.86 millones. Además, se constata que, aunque en algunos casos la construcción puramente voraz ya alcanza valores muy competitivos, la incorporación de aleatoriedad y búsqueda local permite descubrir configuraciones alternativas igualmente óptimas.

#Referencias

[1] Quadratic assignment problem. Wikipedia. Consultado el 2 de febrero de 2026, de https://en.wikipedia.org/wiki/Quadratic_assignment_problem

[2] OpenAI. (2026). ChatGPT (modelo GPT-5). https://chat.openai.com

[3] Reyero, J. (2026). Manual de la asignatura de Algoritmos de Optimización. Universidad Internacional de Valencia.